https://github.com/miladlink/TinyYoloV2

https://github.com/eriklindernoren/PyTorch-YOLOv3


# Setup (Only use the first time or if the dataset was changed)

In [1]:
!python filter_sample_json.py

Found 20 images in the 'data/COCO2017/images/valid_sample' directory.
Successfully loaded original JSON file: 'data/COCO2017/annotations/instances_val2017.json'.
After filtering, 20 image annotations will be kept.
After filtering, 143 annotations will be kept.
Filtering complete! The new JSON file has been saved to: 'data/COCO2017/annotations/instances_val2017_modified_sample.json'
Found 20 images in the 'data/COCO2017/images/train_sample' directory.
Successfully loaded original JSON file: 'data/COCO2017/annotations/instances_train2017.json'.
After filtering, 20 image annotations will be kept.
After filtering, 89 annotations will be kept.
Filtering complete! The new JSON file has been saved to: 'data/COCO2017/annotations/instances_train2017_modified_sample.json'


# Libraries

In [2]:
import os
import time
from PIL import Image
import numpy as np
import json
import cv2
from tqdm import tqdm
# import skimage.io as io
# import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import torch
import torch.optim as optim
import torchvision
from torchvision import transforms
import torchvision.transforms as transforms
from torchvision.datasets.coco import CocoDetection
from torch.utils.data import DataLoader

from utils.YOLOv2 import *
from models.YOLOv3 import load_model
from attacks.FGSM import FGSM
from attacks.PGD import PGD
from attacks.CW import CW
from attacks.noise import Noise
from detect import detect_image
from utils.loss import compute_loss
from utils.utils import load_classes, rescale_boxes, non_max_suppression, print_environment_info
from utils.augmentations import TRANSFORM_TRAIN, TRANSFORM_VAL
from utils.transforms import DEFAULT_TRANSFORMS, Resize, ResizeEval

# Helper functions + vars


In [1]:
def xyxy2xywh(x):
    # Convert nx4 boxes from [x1, y1, x2, y2] to [x, y, w, h] where xy1=top-left, xy2=bottom-right
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[..., 0] = (x[..., 0] + x[..., 2]) / 2  # x center
    y[..., 1] = (x[..., 1] + x[..., 3]) / 2  # y center
    y[..., 2] = x[..., 2] - x[..., 0]  # width
    y[..., 3] = x[..., 3] - x[..., 1]  # height
    return y

def xywh2xyxy(x):
    # Convert nx4 boxes from [x, y, w, h] to [x1, y1, x2, y2] where xy1=top-left, xy2=bottom-right
    y = x.clone() if isinstance(x, torch.Tensor) else np.copy(x)
    y[..., 0] = x[..., 0] - x[..., 2] / 2  # top left x
    y[..., 1] = x[..., 1] - x[..., 3] / 2  # top left y
    y[..., 2] = x[..., 0] + x[..., 2] / 2  # bottom right x
    y[..., 3] = x[..., 1] + x[..., 3] / 2  # bottom right y
    return y

def yolo2json(boxes, img_copy, image_id):
    # * put into coco format of x_min,y_min, width, height, bbox_conf, cls
    # yolo format is x_center, y_center, w, h, bbox_conf, cls_conf, cls
    predictions = []
    for box in boxes:
        x_center, y_center, w, h, conf, cls = box
        x_min = max(0, (x_center - w / 2) * img_copy.shape[3])
        y_min = max(0, (y_center - h / 2) * img_copy.shape[2])
        width = min(img_copy.shape[3], w * img_copy.shape[3])
        height = min(img_copy.shape[2], h * img_copy.shape[2])
        # print(x_min,y_min, width, height, bbox_conf, cls)
        predictions.append({
            'image_id': image_id,
            'category_id': int(id_list[int(cls)]) if modelv == 3 else int(cls),
            'bbox': [int(x_min), int(y_min), int(width), int(height)],
            'score': round(float(conf),2)
        })
    return predictions

def nms2yolo(boxes, img_copy):
    boxes = xyxy2xywh(boxes) # convert from coco to yolo: nms returns nx6 (x1, y1, x2, y2, conf, cls), change to center coordinates [x_center, y_center, width, height]
    boxes[:,0] = boxes[:,0]/img_copy.shape[3]
    boxes[:,1] = boxes[:,1]/img_copy.shape[2]
    boxes[:,2] = boxes[:,2]/img_copy.shape[3]
    boxes[:,3] = boxes[:,3]/img_copy.shape[2]
    return boxes

def saveImageWithBoxes(images, boxes, class_names, fileName):
    to_pil = transforms.ToPILImage()
    pil_image = to_pil(images.squeeze())
    pred_img = plot_boxes(pil_image, boxes, None, class_names)
    pred_img.save(fileName)

def saveImage(img):
    # * just for sanity check, output image. put the dim 3 at the back
    imageN = img.clone().detach()
    imageN = imageN.cpu().squeeze().permute(1, 2, 0).numpy()
    imageN = cv2.cvtColor(imageN, cv2.COLOR_RGB2BGR)
    # print(imageN.shape)
    cv2.imwrite("data/results/mygraph.jpg", imageN*255)

def getOneIter(dataloader):
    images, annotations = next(iter(dataloader))
    np.set_printoptions(linewidth=500)
    np.set_printoptions(suppress=True)
    print("dataloader out")
    print(annotations[0].numpy())


def imgToGreyscale(img):
    if img.shape[0] != 3:
        raise ValueError("Input tensor must have shape [3, H, W].")
    grayscale = 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]
    grayscale_tensor = grayscale.unsqueeze(0).repeat(3, 1, 1)
    return grayscale_tensor

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

os.environ['CUDA_LAUNCH_BLOCKING'] = '1' # reset CUDA debugging environment variable
os.environ['TORCH_USE_CUDA_DSA'] = '1' # enable CUDA DSA for debugging

In [ ]:
epochs = 100 # currently, 100 seems like it works very well
checkpoint_interval = epochs 
# if time is limited then make this smaller, do note that checkpoints are around 270MB per.
modelv = 3
img_size=416

# Model import

In [4]:
# if modelv == 2:
#     model = load_model_v2(weights = './weights/yolov2-tiny-voc.weights').to(device)
#     class_names = ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'TVmonitor']
#     root_train = "./data/VOC2007/JPEGImages"
#     annFile_train = "./data/VOC2007/annotations/train.json"
#     root_val = "./data/VOC2007/JPEGImages"
#     annFile_val = "./data/VOC2007/annotations/val.json"

if modelv == 3:
    model = load_model("./config/yolov3.cfg", "./weights/yolov3.weights")
    class_names = ['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
    id_list = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90])
    root_train = "./data/COCO2017/images/train_sample"
    annFile_train = "./data/COCO2017/annotations/instances_train2017_modified_sample.json"
    root_val = "./data/COCO2017/images/valid_sample"
    annFile_val = "./data/COCO2017/annotations/instances_val2017_modified_sample.json"

else:
    print("invalid model number!")

# COCO loader

create dataloader (make different train and val later)

In [5]:
# coco_dataset_train = CocoDetection(root=root_train, annFile=annFile_train, transform=TRANSFORM_TRAIN_IMG, target_transform=TRANSFORM_TRAIN_TARGET)
# coco_dataset_train = CocoDetection(root=root_train, annFile=annFile_train, transforms=TRANSFORM_TRAIN)
coco_dataset_val = CocoDetection(root=root_val, annFile=annFile_val, transforms=TRANSFORM_VAL)
# coco_dataset_eval = CocoDetection(root=root_val, annFile=annFile_val, transform=transforms.Compose([transforms.ToTensor(),]))

def collate_fn(batch):
    return tuple(zip(*batch))

# Create a DataLoader for your COCO dataset
train_loader = DataLoader(coco_dataset_val, batch_size=4, shuffle=True, collate_fn=collate_fn) # multiple images per batch
val_loader = DataLoader(coco_dataset_val, batch_size=1, shuffle=True, collate_fn=collate_fn)
# one per batch
# cocoeval_loader = DataLoader(coco_dataset_eval, batch_size=1, shuffle=True, collate_fn=collate_fn) # original images without transformatios


loading annotations into memory...
Done (t=0.10s)
creating index...
index created!


In [6]:
getOneIter(val_loader) # print targets

dataloader out
[[1268.           16.            0.30126563    0.516875      0.11676562    0.05223436]
 [1268.            9.            0.19495312    0.36048437    0.21803124    0.02667186]
 [1268.            9.            0.0411875     0.36829687    0.082375      0.02365625]
 [1268.            1.            0.03908984    0.49907813    0.07817969    0.10854688]
 [1268.            1.            0.78387501    0.28696875    0.21612506    0.53704687]
 [1268.            1.            0.62865624    0.48596874    0.10196877    0.13820312]
 [1268.            1.            0.00974609    0.49246875    0.01949219    0.12593749]
 [1268.           77.            0.82642188    0.44768751    0.04637499    0.02929688]
 [1268.           27.            0.03407812    0.52703125    0.03523437    0.0805469 ]
 [1268.           31.            0.76948438    0.47492188    0.16104689    0.35523437]
 [1268.            9.            0.45550001    0.30004687    0.1901094     0.08976562]]


# Adversarial training

In [7]:
eps = 0.05
# attacker = FGSM(model=model, epsilon=0.05)
# attacker = PGD(model=model, epsilon=0.05, epoch=5, lr=0.02)
attacker = CW(model=model, epsilon=eps, lr=eps/3, epoch=5, target=52) # 52 is banana
# attacker = Noise(model=model, epsilon=0.1)


In [8]:
losses = []
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(
            params,
            lr=model.hyperparams['learning_rate'],
            weight_decay=model.hyperparams['decay'],
        )

for epoch in range(1, epochs+1):
    print(f"Starting epoch {epoch}")
    lossesEpoch = []

    for batch_idx, (images, targets) in enumerate(tqdm(train_loader)):
        model.train()

        if targets[0].numel() != 0:
            try:
                #* modify inputs to be in proper shape
                images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
                images = images.to(device)

                # modify targets to be in proper shape
                for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
                    if boxes.ndim == 2:
                        boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss

                targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
                targets = targets[:, :6]

                # verify class indices range
                class_indices = targets[:, 1].long()
                valid_classes = (class_indices >= 0) & (class_indices < 80)

                if not valid_classes.all():
                    print(f"Warning: Invalid class indices found: {class_indices[~valid_classes]}")
                    # Filter out invalid classes
                    targets = targets[valid_classes]
                    if targets.shape[0] == 0:
                        print("No valid targets after filtering, skipping batch")
                        continue

                # ensure all class indices are long
                targets[:, 1] = targets[:, 1].long()

                print(f"Batch {batch_idx}: targets shape: {targets.shape}, class range: {targets[:, 1].min()}-{targets[:, 1].max()}")

                images_adv = attacker.forward(images, targets) # get adversarial image
                outputsBefore = model(images)
                lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
                outputsAfter = model(images_adv)
                lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
                loss = lossBefore + lossAfter

                lossesEpoch.append(loss.detach().cpu().numpy())
                loss.backward()
                optimizer.step()
                # Reset gradients
                optimizer.zero_grad()

                time.sleep(0.1) # for using noise attack

            except RuntimeError as e:
                print(f"Error in batch {batch_idx}: {e}")
                print(f"Targets shape: {targets.shape if 'targets' in locals() else 'undefined'}")
                if 'targets' in locals():
                    print(f"Class indices: {targets[:, 1].unique()}")
                # clear gradients and continue to next batch
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                continue

        else:
            continue # pics without targets

    if lossesEpoch:
        losses_avg = np.average(lossesEpoch)
        print(f"Epoch {epoch} average loss: {losses_avg}")
        losses.append(losses_avg)

    if epoch % checkpoint_interval == 0:
        checkpoint_path = f"./data/results/checkpoints/yolov3_ckpt_{epoch}.pth"
        print(f"---- Saving checkpoint to: '{checkpoint_path}' ----")
        os.makedirs("./data/results/checkpoints", exist_ok=True)
        torch.save(model.state_dict(), checkpoint_path)

Starting epoch 1


  0%|          | 0/5 [00:00<?, ?it/s]

       device='cuda:0')
Batch 0: targets shape: torch.Size([17, 6]), class range: 1.0-76.0


 20%|██        | 1/5 [00:06<00:26,  6.66s/it]

Batch 1: targets shape: torch.Size([24, 6]), class range: 1.0-77.0


...

Batch 3: targets shape: torch.Size([32, 6]), class range: 1.0-77.0


 80%|████████  | 4/5 [00:04<00:01,  1.04s/it]

       device='cuda:0')
Batch 4: targets shape: torch.Size([16, 6]), class range: 1.0-65.0


100%|██████████| 5/5 [00:05<00:00,  1.04s/it]


Epoch 100 average loss: 0.06483554095029831
---- Saving checkpoint to: './data/results/checkpoints/yolov3_ckpt_100.pth' ----


## Load adversarial trained model

In [9]:
if modelv == 3:
    model = load_model("./config/yolov3.cfg", f"./data/results/checkpoints/yolov3_ckpt_{epochs}.pth") # <- this assumes checkpoint interval is a factor of epochs
    class_names = ['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']
    id_list = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90])
    root_train = "./data/COCO2017/images/train_sample"
    annFile_train = "./data/COCO2017/annotations/instances_train2017_modified_sample.json"
    root_val = "./data/COCO2017/images/valid_sample"
    annFile_val = "./data/COCO2017/annotations/instances_val2017_modified_sample.json"

else:
    print("invalid model number!")

# Attack Evaluation

In [10]:
# attackImage = 0 # variable for saving attack image, run this first, change pruning ratio (attack), 
# #don't run this and only run below cells

### NOTE: Attacker was defined here.

In [10]:
predictionsBefore = []
predictionsAfter = []
lossesBefore = []
lossesAfter = []
mode = "image" # need different modes if i want to save image or output prediction json
# mode = "json"
image_ids= [139, 285, 632, 724, 776, 785, 802, 872, 885, 1000,
            1268, 1296, 1353,1425, 1490, 1503, 1532, 1584, 1675, 1761]

os.makedirs("./data/results/images", exist_ok=True)

for i, (images, targets) in enumerate(tqdm(val_loader)):
    if targets[0].numel() != 0:
        with torch.no_grad():
            #* modify inputs to be in proper shape
            images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
            images = images.to(device)
            image_id = int(targets[0][0,0].cpu().numpy()) # assume 1 image
            if image_id not in image_ids: continue # for when we want outputs of specific images
            for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
                if boxes.ndim == 2: boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss. this is normally done in ListDataset -> collate_fn. the id now starts at 0 for each image
            targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
            # originalImageSize = targets[0, 6:].cpu().numpy() # original image shape, assume one image per batch - NOT available in json format
            img_info = coco_dataset_val.coco.imgs[image_id]
            originalImageSize = (img_info['height'], img_info['width'])
            targets = targets[:, :6]

            # print debugging information
            print(f"Image ID: {image_id}")
            print(f"Original targets shape: {targets.shape}")
            print(f"Targets data:")
            print(targets)
            print(f"Class indices: {targets[:, 1]}")
            print(f"Class range: {targets[:, 1].min()} - {targets[:, 1].max()}")

            # check mapping of class indices
            original_classes = targets[:, 1].clone()
            print(f"Original class IDs: {original_classes}")

            # mapping class indices ([1, 80] to [0, 79] range
            targets[:, 1] = targets[:, 1] - 1

            # verify mapped class indices
            mapped_classes = targets[:, 1]
            print(f"Mapped class IDs: {mapped_classes}")
            print(f"Mapped class range: {mapped_classes.min()} - {mapped_classes.max()}")

            # ensure all classes are in range [0, 79]
            valid_mask = (mapped_classes >= 0) & (mapped_classes < 80)
            if not valid_mask.all():
                print(f"Invalid class indices found: {mapped_classes[~valid_mask]}")
                targets = targets[valid_mask]
                if targets.shape[0] == 0:
                    print("No valid targets after filtering, skipping image")
                    continue
                print(f"Filtered targets shape: {targets.shape}")

            # ensure all class indices are long
            targets[:, 1] = targets[:, 1].long()

            # final validation
            final_classes = targets[:, 1]
            print(f"Final class indices: {final_classes}")
            print(f"Final class range: {final_classes.min()} - {final_classes.max()}")
            print(f"All classes in range [0, 79]: {((final_classes >= 0) & (final_classes < 80)).all()}")

            #* loss
            model.train()
            try:
                outputsBefore = model(images)
                print(f"Model output shapes: {[out.shape for out in outputsBefore]}")

                lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
                lossesBefore.append(lossBefore.cpu().numpy())

                images_adv = attacker.forward(images, targets) # get adversarial image

                outputsAfter = model(images_adv)
                lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
                lossesAfter.append(lossAfter.cpu().numpy())

            except RuntimeError as e:
                print(f"CUDA error occurred: {e}")
                print(f"Error details:")
                print(f"  Targets shape: {targets.shape}")
                print(f"  Class indices: {targets[:, 1]}")
                print(f"  Class unique values: {targets[:, 1].unique()}")
                print(f"  Class data type: {targets[:, 1].dtype}")

                # clean CUDA cache and skip this iteration
                torch.cuda.empty_cache()
                continue

            #* plot
            model.eval()

            # before attack
            outputsBefore = model(images[0].unsqueeze(0))
            boxesBefore = non_max_suppression(outputsBefore, conf_thres=0.3, iou_thres=0.5)[0].numpy()
            if mode == "json":
                boxesBefore = rescale_boxes(boxesBefore, img_size, originalImageSize)
            boxesBefore = nms2yolo(boxesBefore, images)
            if mode == "image":
                saveImageWithBoxes(images[0], boxesBefore, class_names, f"./data/results/images/attack_before_{image_id}.jpg")
            if mode == "json":
                predictionsBefore += yolo2json(boxesBefore, images[0].unsqueeze(0), image_id)

            # after attack
            outputsAfter = model(images_adv[0].unsqueeze(0))
            boxesAfter = non_max_suppression(outputsAfter, conf_thres=0.3, iou_thres=0.5)[0].numpy()

            if mode == "json":
                boxesAfter = rescale_boxes(boxesAfter, img_size, originalImageSize)
            boxesAfter = nms2yolo(boxesAfter, images_adv)
            print(boxesAfter)
            if mode == "image":
                saveImageWithBoxes(images_adv[0], boxesAfter, class_names, f"./data/results/images/attack_after_{image_id}.jpg")
            if mode == "json":
                predictionsAfter += yolo2json(boxesAfter, images_adv[0].unsqueeze(0), image_id)

    else: continue # pics without targets

with open(f'./data/results/predictionsBefore.json', 'w') as f:
    json.dump(predictionsBefore, f)
with open(f'./data/results/predictionsAfter.json', 'w') as f:
    json.dump(predictionsAfter, f)
np.savetxt("./data/results/lossesBefore.csv", lossesBefore, delimiter=",")
np.savetxt("./data/results/lossesAfter.csv", lossesAfter, delimiter=",")

  0%|          | 0/20 [00:00<?, ?it/s]

Image ID: 1761
Original targets shape: torch.Size([7, 6])
Targets data:
tensor([[0.0000, 5.0000, 0.6066, 0.2178, 0.0800, 0.0684],
        [0.0000, 5.0000, 0.4003, 0.0369, 0.1158, 0.0738],
        [0.0000, 1.0000, 0.1838, 0.9624, 0.0076, 0.0180],
        [0.0000, 1.0000, 0.1723, 0.9630, 0.0113, 0.0153],
        [0.0000, 1.0000, 0.1900, 0.9609, 0.0097, 0.0201],
        [0.0000, 1.0000, 0.2222, 0.9777, 0.0075, 0.0113],
        [0.0000, 1.0000, 0.1989, 0.9642, 0.0115, 0.0166]], device='cuda:0',
       dtype=torch.float64)
Class indices: tensor([5., 5., 1., 1., 1., 1., 1.], device='cuda:0', dtype=torch.float64)
Class range: 1.0 - 5.0
Original class IDs: tensor([5., 5., 1., 1., 1., 1., 1.], device='cuda:0', dtype=torch.float64)
Mapped class IDs: tensor([4., 4., 0., 0., 0., 0., 0.], device='cuda:0', dtype=torch.float64)
Mapped class range: 0.0 - 4.0
Final class indices: tensor([4., 4., 0., 0., 0., 0., 0.], device='cuda:0', dtype=torch.float64)
Final class range: 0.0 - 4.0
All classes in range

  5%|▌         | 1/20 [00:01<00:24,  1.27s/it]

[[0.39982685 0.03593565 0.12298378 0.07030621 0.94321024 5.        ]
 [0.6056099  0.21702789 0.0821045  0.06966686 0.9058492  5.        ]
 [0.18172625 0.9645237  0.00970422 0.01652189 0.50017023 1.        ]
 [0.19895214 0.967426   0.0107233  0.01474923 0.32698664 1.        ]
 [0.16992714 0.9633322  0.0119955  0.01617945 0.3181952  1.        ]]
Image ID: 1490
Original targets shape: torch.Size([2, 6])
Targets data:
tensor([[0.0000e+00, 1.0000e+00, 7.0136e-01, 4.3806e-01, 7.9531e-02, 1.9192e-01],
        [0.0000e+00, 4.2000e+01, 5.6128e-01, 6.1295e-01, 3.4202e-01, 2.3484e-02]],
       device='cuda:0', dtype=torch.float64)
Class indices: tensor([ 1., 42.], device='cuda:0', dtype=torch.float64)
Class range: 1.0 - 42.0
Original class IDs: tensor([ 1., 42.], device='cuda:0', dtype=torch.float64)
Mapped class IDs: tensor([ 0., 41.], device='cuda:0', dtype=torch.float64)
Mapped class range: 0.0 - 41.0
Final class indices: tensor([ 0., 41.], device='cuda:0', dtype=torch.float64)
Final class ran

 10%|█         | 2/20 [00:01<00:16,  1.07it/s]

...

[[ 0.78547126  0.28420255  0.23377755  0.60654616  0.97697717  1.        ]
 [ 0.62587667  0.4886036   0.09817244  0.13704425  0.9683974   1.        ]
 [ 0.03945085  0.49820724  0.07482727  0.10563719  0.95099455  1.        ]
 [ 0.7717958   0.47590834  0.14710602  0.33935723  0.92560935 31.        ]
 [ 0.04083711  0.36924186  0.07815227  0.02723973  0.9154509   9.        ]
 [ 0.45414183  0.29839796  0.21051846  0.12508528  0.9022528   9.        ]
 [ 0.00936288  0.49229145  0.01898784  0.11508303  0.89993614  1.        ]
 [ 0.03469241  0.52640605  0.03537514  0.08631156  0.891027   27.        ]
 [ 0.29983473  0.5191833   0.11811255  0.05379647  0.6385431  16.        ]
 [ 0.8260733   0.446854    0.04857679  0.02871814  0.34591973 77.        ]]
Image ID: 1353
Original targets shape: torch.Size([7, 6])
Targets data:
tensor([[0.0000, 7.0000, 0.2566, 0.6472, 0.3798, 0.2786],
        [0.0000, 1.0000, 0.5532, 0.3090, 0.1330, 0.2582],
        [0.0000, 1.0000, 0.4218, 0.2730, 0.1510, 0.1052],
   

 95%|█████████▌| 19/20 [00:12<00:00,  1.56it/s]

[[0.41754186 0.2730779  0.14713794 0.11603867 0.9673869  1.        ]
 [0.3970863  0.39956763 0.23999456 0.39305034 0.96688724 1.        ]
 [0.24990566 0.43034342 0.23407787 0.3332534  0.96272093 1.        ]
 [0.5492866  0.31264764 0.11609488 0.25225168 0.92055196 1.        ]
 [0.502331   0.36504763 0.06203879 0.13100693 0.91221786 1.        ]
 [0.26061374 0.6454878  0.37765694 0.26410484 0.88872    7.        ]
 [0.39496008 0.35902125 0.1439099  0.14300469 0.8188536  1.        ]]
Image ID: 785
Original targets shape: torch.Size([2, 6])
Targets data:
tensor([[ 0.0000,  1.0000,  0.4387,  0.2540,  0.3417,  0.5079],
        [ 0.0000, 35.0000,  0.3208,  0.7331,  0.6402,  0.0597]],
       device='cuda:0', dtype=torch.float64)
Class indices: tensor([ 1., 35.], device='cuda:0', dtype=torch.float64)
Class range: 1.0 - 35.0
Original class IDs: tensor([ 1., 35.], device='cuda:0', dtype=torch.float64)
Mapped class IDs: tensor([ 0., 34.], device='cuda:0', dtype=torch.float64)
Mapped class range: 0.0

100%|██████████| 20/20 [00:13<00:00,  1.47it/s]

[[ 0.43867752  0.2551596   0.34673882  0.47893122  0.9522174   1.        ]
 [ 0.33047596  0.7336816   0.48052728  0.08030628  0.5626383  35.        ]]


In [17]:
# predictionsBefore = []
# predictionsAfter = []
# lossesBefore = []
# lossesAfter = []
# # mode = "image" # need different modes if i want to save image or output prediction json
# mode = "json"
# # image_ids= [71711,19221,22192] # output images that i want, 19221 is broccoli, 22192 is dog, 71711 is plane
# # image_ids= [139, 285, 632, 724, 776, 785, 802, 872, 885, 1000,
# #             1268, 1296, 1353,1425, 1490, 1503, 1532, 1584, 1675, 1761] # sample image id
# image_ids = [139]

# os.makedirs("./data/results/images", exist_ok=True)

# for i, (images, targets) in enumerate(tqdm(val_loader)):
#     if targets[0].numel() != 0:
#         with torch.no_grad():
#             #* modify inputs to be in proper shape
#             images = torch.stack(images) # images.shape is [n, 3, 416, 416] (even if n=1)
#             images = images.to(device)
#             image_id = int(targets[0][0,0].cpu().numpy()) # assume 1 image
#             if image_id not in image_ids: continue # for when we want outputs of specific images
#             for i, boxes in enumerate(targets): # targets is nx6, (image,class,x,y,w,h)
#                 if boxes.ndim == 2: boxes[:, 0] = i # change out image_id to id in batch to conform to compute_loss. this is normally done in ListDataset -> collate_fn. the id now starts at 0 for each image
#             targets = torch.cat(targets, 0).to(device) # from tuples to one tensor
#             # originalImageSize = targets[0, 6:].cpu().numpy() # original image shape, assume one image per batch - NOT available in json format
#             img_info = coco_dataset_val.coco.imgs[image_id]
#             originalImageSize = (img_info['height'], img_info['width'])
#             targets = targets[:, :6]

#             #* loss
#             model.train()
#             # start = time.time()
#             outputsBefore = model(images)
#             # end = time.time()
#             # print(end - start)
#             lossBefore, loss_components = compute_loss(outputsBefore, targets, model)
#             lossesBefore.append(lossBefore.cpu().numpy())

#             images_adv = attacker.forward(images, targets) # get adversarial image

#             outputsAfter = model(images_adv)
#             lossAfter, loss_components = compute_loss(outputsAfter, targets, model)
#             lossesAfter.append(lossAfter.cpu().numpy())

#             #* plot
#             model.eval()

#             # ground truth
#             # print(targets) #(ima ge,class,x,y,w,h), the class id starts from 1
#             # nms is (x1, y1, x2, y2, conf, cls), the class id starts from 0
#             # yolo is (x_center, y_center, width, height, conf. cls)

#             # before attack
#             outputsBefore = model(images[0].unsqueeze(0))
#             boxesBefore = non_max_suppression(outputsBefore, conf_thres=0.3, iou_thres=0.5)[0].numpy()
#             if mode == "json":
#                 boxesBefore = rescale_boxes(boxesBefore, img_size, originalImageSize)
#             boxesBefore = nms2yolo(boxesBefore, images)
#             if mode == "image":
#                 saveImageWithBoxes(images[0], boxesBefore, class_names, f"./data/results/images/attack_before_{image_id}.jpg")
#             if mode == "json":
#                 predictionsBefore += yolo2json(boxesBefore, images[0].unsqueeze(0), image_id)

#             # after attack
#             outputsAfter = model(images_adv[0].unsqueeze(0))
#             boxesAfter = non_max_suppression(outputsAfter, conf_thres=0.3, iou_thres=0.5)[0].numpy()


#             if mode == "json":
#                 boxesAfter = rescale_boxes(boxesAfter, img_size, originalImageSize)
#             # print(boxesAfter)
#             boxesAfter = nms2yolo(boxesAfter, images_adv)
#             print(boxesAfter)
#             if mode == "image":
#                 saveImageWithBoxes(images_adv[0], boxesAfter, class_names, f"./data/results/images/attack_after_{image_id}.jpg")

#                 # attackImage = images_adv[0] # for saving the same attack image for different pruning ratios, comment out after save
#                 # saveImageWithBoxes(attackImage, boxesAfter, class_names, f"./data/results/images/pruning/{image_id}/attack_after_99_x.jpg") # plot different pruning ratios with same attack image

#                 # greyscaleAttackImage = imgToGreyscale(attackImage)
#                 # saveImageWithBoxes(greyscaleAttackImage, boxesAfter, class_names, f"./data/results/images/pruning/{image_id}/attack_after_x_grey.jpg") # plot different pruning ratios with same attack image
#             if mode == "json":
#                 predictionsAfter += yolo2json(boxesAfter, images_adv[0].unsqueeze(0), image_id)
#             # time.sleep(0.1) # for using noise attack

#     else: continue # pics without targets
#     # break


# with open(f'./data/results/predictionsBefore.json', 'w') as f:
#     json.dump(predictionsBefore, f)
# with open(f'./data/results/predictionsAfter.json', 'w') as f:
#     json.dump(predictionsAfter, f)
# np.savetxt("./data/results/lossesBefore.csv", lossesBefore, delimiter=",")
# np.savetxt("./data/results/lossesAfter.csv", lossesAfter, delimiter=",")

  0%|          | 0/20 [00:00<?, ?it/s]


RuntimeError: CUDA error: device-side assert triggered
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# data = np.loadtxt('./data/results/lossesBefore.csv', delimiter=',')
# average = np.mean(data)
# print("Avg loss before attack:", average)
# data = np.loadtxt('./data/results/lossesAfter.csv', delimiter=',')
# average = np.mean(data)
# print("Avg loss after attack:", average)

# Get mAP

In [ ]:
# from pycocotools.coco import COCO
# from pycocotools.cocoeval import COCOeval

# coco_gld = COCO(annFile_val) # coco
# # if modelv == 2:
# #     coco_rst = coco_gld.loadRes('./data/results/v2predictions.json')
# # elif modelv == 3:
# #     coco_rst = coco_gld.loadRes('./data/results/v3predictions.json')

# coco_rst = coco_gld.loadRes('./data/results/predictionsAfter.json')
# cocoEval = COCOeval(coco_gld, coco_rst, iouType='bbox')
# cocoEval.evaluate()
# cocoEval.accumulate()
# cocoEval.summarize()